# Shape: profile a large lakehouse table across the Spark cluster (PySpark notebook)

Attach the **Shape Environment** (`integrations/fabric/environment/README.md`) and the default
lakehouse; Shape must be installed on the executors, which the Environment does.

`mode = "distributed"` (the default): every partition is profiled on the executors in Shape's
bounded mode (sketches, memory independent of the partition's size) through `mapInArrow`, and the
driver merges the partial profiles. No rows reach the driver. The result is a bounded profile:
counts, min, max, mean, variance and null counts are exact; distinct counts, quantiles and top
values carry the error bounds in the profile's `error_models`. The merge is in partition order.

`mode = "exact"`: the driver-only exact profile of `shape_profile_spark`, for tables that fit in
driver memory. Only this mode can check a contract or diff against a baseline.

The exit value has the keys of `shape_profile.ipynb`, plus `mode` and `checked`.
**`checked` is false for the distributed mode**: nothing was checked against a contract, so
do not use it as a gate; use `mode = "exact"` for gates.


In [ ]:
# Parameters cell (toggle "parameter cell" in Fabric). A pipeline overrides these.
# Paths are relative to the lakehouse Files/ folder, or absolute.
tableName = "orders"  # Delta table in the default lakehouse ("schema.table" also accepted)
contractPath = ""  # optional contract JSON, for example "contracts/orders.json"
baselinePath = ""  # optional earlier .shape artifact to diff against
outputDir = "shape"  # artifacts go to Files/<outputDir>/<table>/<timestamp>/
failOnDrift = False  # True: drift against the baseline also fails the gate
mode = "distributed"  # "distributed": bounded profile per partition, merged on the driver
# "exact": driver-only exact profile (table must fit in driver memory)
partitions = 0  # distributed only: repartition first (0 keeps the table's partitioning)

In [ ]:
import inspect
import json
import math
import os
from datetime import UTC, datetime
from pathlib import Path

import pyarrow as pa

import shape
from shape.kernel.dispatch import get_kernel

KERNEL = get_kernel().NAME  # "rust" with a platform wheel, "python" with the pure-Python wheel
print(f"Shape {shape.__version__}, kernel: {KERNEL}")

LAKEHOUSE = "/lakehouse/default"
FILES = f"{LAKEHOUSE}/Files"


def _as_bool(value) -> bool:
    """Pipeline parameters may arrive as strings."""
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


def _resolve(path: str) -> str:
    return path if os.path.isabs(path) else f"{FILES}/{path}"


failOnDrift = _as_bool(failOnDrift)
safe_name = str(tableName).replace(".", "_")
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
out_rel = f"{outputDir.strip('/')}/{safe_name}/{stamp}"
out_dir = Path(FILES) / out_rel
MAX_LISTED = 100  # keep the exit value small (well under 1 MB)
mode = str(mode).strip().lower()
if mode not in ("distributed", "exact"):
    raise ValueError(f"mode must be 'distributed' or 'exact', got {mode!r}")
if mode == "distributed" and (contractPath or baselinePath):
    raise ValueError(
        "contractPath and baselinePath need the exact profile: the distributed mode builds a "
        "bounded profile, which the contract check and the diff do not read. "
        "Set mode = 'exact', or clear both parameters."
    )


def _json_safe(value):
    """NaN and infinities become null so the artifact is strict JSON."""
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {k: _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    return value


# Exact mode only: the Arrow copy of the table lives in driver memory (see shape_profile_spark).
DRIVER_ROW_LIMIT = 5_000_000
SAMPLE_SEED = 42

In [ ]:
# `spark` is predefined in Fabric
df = spark.read.table(str(tableName))  # noqa: F821
print(f"Spark {spark.version}, {df.rdd.getNumPartitions()} partition(s)")  # noqa: F821

if mode == "exact":
    total_rows = df.count()
    sampled = total_rows > DRIVER_ROW_LIMIT
    if sampled:
        df = df.sample(
            withReplacement=False, fraction=DRIVER_ROW_LIMIT / total_rows, seed=SAMPLE_SEED
        )
    if int(spark.version.split(".")[0]) >= 4:  # noqa: F821
        table = df.toArrow()
    else:
        table = pa.Table.from_pandas(df.toPandas(), preserve_index=False)
    kwargs = {"name": str(tableName)}
    if "exact" in inspect.signature(shape.profile).parameters:
        kwargs["exact"] = True
    profile = shape.profile(table, **kwargs)
    print(f"Exact profile of {table.num_rows:,} of {total_rows:,} rows (sampled={sampled})")
else:
    from shape.integrations.fabric.spark import profile_distributed

    doc = profile_distributed(df, name=str(tableName), partitions=int(partitions) or None)
    entry = doc["tables"][str(tableName)]
    total_rows = entry["rows"]
    sampled = False  # every row of every partition is read
    print(f"Bounded profile of {total_rows:,} rows x {len(entry['columns'])} columns")

In [ ]:
violations, changes, drifted = [], [], False

if mode == "exact":
    if contractPath:
        violations = list(shape.check(profile, _resolve(contractPath)).violations)
    if baselinePath:
        diff_result = shape.diff(shape.load(_resolve(baselinePath)), profile)
        drifted = bool(diff_result.drifted)
        changes = list(diff_result.changes)
        if drifted and failOnDrift:
            violations.append(
                {
                    "column": "*",
                    "rule": "drift",
                    "expected": "no drift against baseline",
                    "observed": f"{len(changes)} change(s)",
                }
            )

passed = not violations

In [ ]:
SUMMARY_KEYS = (
    "arrow_type",
    "kind",
    "count",
    "null_count",
    "distinct",
    "min",
    "max",
    "mean",
)
out_dir.mkdir(parents=True, exist_ok=True)
if mode == "exact":
    shape.save(profile, str(out_dir / f"{safe_name}.shape"))
    (out_dir / f"{safe_name}.html").write_text(profile.to_html(), encoding="utf-8")
    (out_dir / f"{safe_name}.summary.json").write_text(
        json.dumps(profile.summary()), encoding="utf-8"
    )
    artifact_path = f"{out_rel}/{safe_name}.shape"
else:
    # the full bounded profile document, plus a one-line-per-column summary
    (out_dir / f"{safe_name}.profile.json").write_text(
        json.dumps(_json_safe(doc)), encoding="utf-8"
    )
    summary = {
        "name": entry["name"],
        "rows": entry["rows"],
        "mode": "bounded",
        "columns": {c["name"]: {k: c.get(k) for k in SUMMARY_KEYS} for c in entry["columns"]},
    }
    (out_dir / f"{safe_name}.summary.json").write_text(
        json.dumps(_json_safe(summary)), encoding="utf-8"
    )
    artifact_path = f"{out_rel}/{safe_name}.profile.json"
print("Artifacts written to", out_dir)

In [ ]:
if mode == "exact":
    from IPython.display import HTML, display

    display(HTML(profile.to_html()))
else:
    print(json.dumps(_json_safe(summary), indent=1)[:4000])

In [ ]:
result = {
    "table": tableName,
    "rows": total_rows,
    "passed": passed,
    "violations": violations[:MAX_LISTED],
    "drifted": drifted,
    "changes": changes[:MAX_LISTED],
    "artifactPath": artifact_path,
    "sampled": sampled,
    "truncated": len(violations) > MAX_LISTED or len(changes) > MAX_LISTED,
    "kernel": KERNEL,
    "mode": "bounded" if mode == "distributed" else "exact",
    "checked": mode == "exact",
}
print(json.dumps(result, indent=2, default=str)[:4000])

In [ ]:
import notebookutils

notebookutils.notebook.exit(json.dumps(result, default=str))